In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib
matplotlib.rcParams["figure.figsize"] = (20,10)

In [ ]:
df1 = pd.read_csv("bengaluru_house_prices.csv")

df1.head()

In [ ]:
df1.shape

In [ ]:
df1.columns

In [ ]:
df1['area_type'].unique()

In [ ]:
df1['area_type'].value_counts()

In [ ]:
df2 = df1.drop(columns = ['area_type', 'availability', 'society', 'balcony' ])

df2.shape

In [ ]:
df2.isnull().sum()

In [ ]:
df3 = df2.dropna()
df3.isnull().sum()

In [ ]:
df3.shape

In [ ]:
df3['bhk'] = df3['size'].apply(lambda x : int(x.split(' ')[0]))

df3['bhk'].unique()

In [ ]:
def is_float(x):
    try:
        float(x)
    except:
        return False
    return True

In [ ]:
df3[~df3['total_sqft'].apply(is_float)].head()

In [ ]:
def convert_dqft_to_num(x):
    tokens = x.split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1]))/2
    try:
        return float(x)
    except:
        return None

In [ ]:
df4 = df3.copy()
df4['total_sqft'] = df4['total_sqft'].apply(convert_dqft_to_num)
df4 = df4[df4['total_sqft'].notnull()]

df4

In [ ]:
df5 = df4.copy()
df5['price_per_sqft'] = df5['price']*100000/df5['total_sqft']

df5.head()

In [ ]:
df5['price_per_sqft'].describe()

In [ ]:
df5['location'] = df5['location'].apply(lambda x : x.strip())
location_stats = df5['location'].value_counts(ascending=False)
location_stats

In [ ]:
location_stats_less_than_10 = location_stats[location_stats<=10]
location_stats_less_than_10

In [ ]:
df5['location'] = df5['location'].apply(lambda x: 'other' if x in location_stats_less_than_10 else x)
len(df5['location'].unique())

In [ ]:
df5.head(10)

In [ ]:
df5[df5['total_sqft']/df5['bhk']<300]

In [ ]:
df6 = df5[~(df5['total_sqft']/df5['bhk']<300)]

df6.shape

In [ ]:
df6['price_per_sqft'].describe()

In [ ]:
def remove_pps_outliers(df):
    df_out = pd.DataFrame()
    for key, subdf in df.groupby('location'):
        m = np.mean(subdf.price_per_sqft)
        st =  np.std(subdf.price_per_sqft)
        reduce_df = subdf[(subdf.price_per_sqft>(m-st)) & (subdf.price_per_sqft<=(m+st))]
        df_out = pd.concat([df_out, reduce_df], ignore_index=True)
    return df_out

df7 = remove_pps_outliers(df6)
df7.shape

In [ ]:
def plot_scatter_chart(df, location):
    bhk2 = df[(df['location'] == location) & (df['bhk']==2)]
    bhk3 = df[(df['location'] == location) & (df['bhk']==3)]
    matplotlib.rcParams['figure.figsize'] = (15,10)
    plt.scatter(bhk2['total_sqft'], bhk2['price'], color = 'blue', label = '2 BHK', s = 50)
    plt.scatter(bhk3['total_sqft'], bhk3['price'], color = 'red', label = '3 BHK', s = 50)
    plt.xlabel("Area")
    plt.ylabel("Price")
    plt.title(location)
    plt.legend()

plot_scatter_chart(df7, "Rajaji Nagar")

In [ ]:
plot_scatter_chart(df7, "Hebbal")

In [ ]:
def remove_bhk_outlier(df):
    exclusive_indices = np.array([])
    for location, location_df in df.groupby('location'):
        bhk_stats = {}
        for bhk, bhk_df in location_df.groupby('bhk'):
            bhk_stats[bhk] = {
                'mean' : np.mean(bhk_df['price_per_sqft']),
                'std' : np.std(bhk_df['price_per_sqft']),
                'count' : bhk_df.shape[0]
            }
        for bhk, bhk_df in location_df.groupby('bhk'):
            stats = bhk_stats.get(bhk-1)
            if stats and stats['count']>5:
                exclusive_indices = np.append(exclusive_indices, bhk_df[bhk_df['price_per_sqft']<stats['mean']].index.values)
    return df.drop(exclusive_indices, axis = 'index')

df8 = remove_bhk_outlier(df7)
df8.shape

In [ ]:
plot_scatter_chart(df8, "Rajaji Nagar")

In [ ]:
df8['bath'].unique()

In [ ]:
plt.hist(df8['bath'], rwidth=0.8)
plt.xlabel('Number of bathroom')
plt.ylabel('Count')

In [ ]:
df8[df8['bath']>10]

In [ ]:
df8[df8['bath']>df8['bhk']+2]

In [ ]:
df9 = df8[df8['bath']<df8['bhk']+2]
df9.shape

In [ ]:
df10 = df9.drop(columns = ['size', 'price_per_sqft'])
df10

In [ ]:
dummies =  pd.get_dummies(df10['location']).astype(int)
dummies.head(2)

In [ ]:
df11 = pd.concat([df10, dummies.drop('other', axis = 'columns')], axis = 'columns')
df11.head()

In [ ]:
df12 = df11.drop('location', axis = 'columns')
df12.head()

In [ ]:
x = df12.drop(['price'], axis = 'columns')
x

In [ ]:
y = df12['price']
y

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=10)

In [ ]:
from sklearn.linear_model import LinearRegression
lr_model = LinearRegression()
lr_model.fit(x_train, y_train)
lr_model.score(x_test, y_test)

In [ ]:
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_val_score

cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=0)

cross_val_score(LinearRegression(), x, y, cv = cv)

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor

#for Lasso
cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=0)

cross_val_score(Lasso(), x, y, cv = cv)

In [ ]:
#for Decision Tree
cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=0)

cross_val_score(DecisionTreeRegressor(), x, y, cv = cv)

In [ ]:
#Make a function of predict price

def predict_price(location, sqft, bath, bhk):
    loc_index = np.where(x.columns == location)[0][0]

    X = np.zeros(len(x.columns))

    X[0] = sqft
    X[1] = bath
    X[2] = bhk
    if loc_index>=0:
        X[loc_index] = 1

    return lr_model.predict([X])[0]

In [ ]:
x = predict_price('1st Phase JP Nagar', 1000,2,2)

In [ ]:
x